# Experimento: conjunto cooperante $K$ vs First-Fit

Para cada cenário base roda:

- **Cooperação** (`cenario_disaster_aware{i}.pkl`, roteador ponderado disaster-aware): todos os $2^I$ conjuntos $K$, de $K=\emptyset$ a $K=\mathcal{I}$. Com $I=5$: 32 rodadas.
- **First-Fit** (`cenario_nao_disaster_aware{i}.pkl`): mesma topologia, desastre e tráfego, roteado com `FirstFitSubnet`. Ignora cooperação, então roda uma vez.

Métricas de `overleaf/problema.tex`:

| Métrica | Definição |
|---|---|
| $B_n$, $B_m$ | bloqueio de fundo / de migração ($A = 1 - B$) |
| $\eta_i$ | $M_i(t_d)/D_i$: banda de migração aceita antes de $t_d$ sobre o tamanho do datacenter (limitada a 1) |
| $\eta(K)$ | média de $\eta_i$ nos cooperantes — objetivo primário |
| $\Gamma(K)$ | $\eta(K) - \eta(\mathcal{I}\setminus K)$; só com os dois lados não vazios |

**Como rodar:** execute as células em ordem. Cada rodada leva alguns minutos; cada uma grava um `metrics_*.json` em `output/cooperation_experiment_v2/`. Se interromper, rode de novo: rodadas prontas são puladas.

**Cuidados de leitura**

- $|K|=1$ deve dar o mesmo que $K=\emptyset$ (com um só ISP não há o que compartilhar). É um controle.
- **v2:** o roteador ponderado agora tenta os caminhos na ordem calculada com os pesos α/β/γ da cooperação (antes ele reordenava por pesos fixos e descartava essa ordem). Os resultados da v1 ficam em `output/cooperation_experiment/`.
- Cooperar só troca informação para os pesos; não reserva espectro nem dá prioridade. $\Gamma \approx 0$ continua sendo um resultado válido.

In [ ]:
import json
import os
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Workers live in .py files so multiprocessing can pickle them
from experiment_worker import run_cooperation_experiment
from run_cooperation_experiment import consolidar, montar_tarefas

%matplotlib inline

## Parâmetros

In [ ]:
BASE_INDICES = [0]            # e.g. list(range(10)) for all base scenarios
INCLUIR_FIRST_FIT = True
# v2: router keeps the cooperation (α/β/γ) path order. v1 results stay in
# output/cooperation_experiment for comparison.
OUTPUT_DIR = Path("output/cooperation_experiment_v2")
MAX_WORKERS = min(os.cpu_count() or 4, 8)   # each process holds one scenario (~1 GB RAM)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
tarefas = montar_tarefas(BASE_INDICES, OUTPUT_DIR, INCLUIR_FIRST_FIT)
for idx, t in enumerate(tarefas):
    t.update(output_dir=str(OUTPUT_DIR), run_index=idx, total_runs=len(tarefas))
print(f"{len(tarefas)} rodadas pendentes | bases {BASE_INDICES} | {MAX_WORKERS} processos")

## Rodar

O progresso aparece abaixo conforme as rodadas terminam.

In [ ]:
inicio = datetime.now()
falhas = []
if tarefas:
    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(run_cooperation_experiment, t) for t in tarefas]
        for n, future in enumerate(as_completed(futures), start=1):
            r = future.result()
            estado = "✓" if r["success"] else "✗"
            print(f"[{n}/{len(tarefas)}] {estado} base={r['base_index']} {r['K']}  ({datetime.now() - inicio})")
            if not r["success"]:
                falhas.append(r)
print(f"Duração: {datetime.now() - inicio} | falhas: {len(falhas)}")
consolidar(OUTPUT_DIR)

## Carregar resultados

In [ ]:
resumo_df = pd.read_csv(OUTPUT_DIR / "resumo_por_rodada.csv")
por_isp_df = pd.read_csv(OUTPUT_DIR / "resumo_por_isp.csv")
resumo_df = resumo_df[resumo_df["base_index"].isin(BASE_INDICES)]
por_isp_df = por_isp_df[por_isp_df["base_index"].isin(BASE_INDICES)]

coop = resumo_df[resumo_df["variante"] == "cooperacao"]
ff = resumo_df[resumo_df["variante"] == "first_fit"]
isp_ids = sorted(por_isp_df["isp_id"].unique())
I = len(isp_ids)
print(f"{len(coop)} rodadas de cooperação | {len(ff)} de First-Fit | I = {I}")

## Cooperação total, nenhuma e First-Fit

In [ ]:
colunas = ["eta_todos", "B_m", "B_n", "B_n_janela"]
comparacao = pd.concat([
    ff.assign(cenario="First-Fit")[["base_index", "cenario"] + colunas],
    coop[coop["k"] == 0].assign(cenario="Ponderado, K = ∅")[["base_index", "cenario"] + colunas],
    coop[coop["k"] == I].assign(cenario="Ponderado, K = I")[["base_index", "cenario"] + colunas],
])
comparacao.groupby("cenario", sort=False)[colunas].mean().round(4)

## Por tamanho de $K$

In [ ]:
metricas = ["eta_K", "eta_isolados", "Gamma", "eta_todos", "B_m", "B_n", "B_n_janela"]
coop.groupby("k")[metricas].agg(["mean", "std", "min", "max"]).round(4)

In [ ]:
COR_K, COR_ISO, COR_FF = "#2a78d6", "#eb6834", "#52514e"
TEXTO, GRADE = "#52514e", "#e5e4df"

def estilo(ax, titulo, ylabel):
    ax.set_title(titulo, loc="left", fontsize=12, color="#0b0b0b")
    ax.set_xlabel("|K| (provedores cooperando)", color=TEXTO)
    ax.set_ylabel(ylabel, color=TEXTO)
    ax.set_xticks(range(I + 1))
    ax.tick_params(colors=TEXTO)
    ax.grid(axis="y", color=GRADE, linewidth=0.8)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: every run is a dot; the line is the mean per |K|
ax = axes[0]
for coluna, cor, nome, desloc in [("eta_K", COR_K, "η(K) cooperantes", -0.08),
                                  ("eta_isolados", COR_ISO, "η(I∖K) isolados", 0.08)]:
    dados = coop.dropna(subset=[coluna])
    ax.scatter(dados["k"] + desloc, dados[coluna], s=36, color=cor, alpha=0.35,
               edgecolors="white", linewidths=1)
    media = dados.groupby("k")[coluna].mean()
    ax.plot(media.index + desloc, media.values, color=cor, linewidth=2, marker="o",
            markersize=8, label=nome)
if len(ff):
    ax.axhline(ff["eta_todos"].mean(), color=COR_FF, linewidth=1.5, linestyle="--",
               label="First-Fit (todos)")
estilo(ax, "Fração migrada por grupo", "η")
ax.legend(frameon=False, loc="best")

ax = axes[1]
dados = coop.dropna(subset=["Gamma"])
ax.axhline(0, color=TEXTO, linewidth=1)
ax.scatter(dados["k"], dados["Gamma"], s=36, color=COR_K, alpha=0.35,
           edgecolors="white", linewidths=1)
media = dados.groupby("k")["Gamma"].mean()
ax.plot(media.index, media.values, color=COR_K, linewidth=2, marker="o", markersize=8)
estilo(ax, "Γ(K) = η(K) − η(I∖K)", "Γ")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "eta_gamma_por_k.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Blocking (secondary objective): one chart per metric, no dual axis
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, coluna, titulo in [(axes[0], "B_n", "Bloqueio de fundo B_n"),
                           (axes[1], "B_m", "Bloqueio de migração B_m")]:
    media = coop.groupby("k")[coluna].mean()
    ax.plot(media.index, media.values, color=COR_K, linewidth=2, marker="o",
            markersize=8, label="Ponderado (média por |K|)")
    if len(ff):
        ax.axhline(ff[coluna].mean(), color=COR_FF, linewidth=1.5, linestyle="--",
                   label="First-Fit")
    estilo(ax, titulo, "taxa de bloqueio")
    ax.legend(frameon=False, loc="best")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "bloqueio_por_k.png", dpi=150, bbox_inches="tight")
plt.show()

## Mesmo provedor dentro e fora de $K$

Para cada ISP: $\eta_i$ médio quando coopera menos quando está isolado (só $0 < |K| < I$). First-Fit ao lado como referência.

In [ ]:
pc = por_isp_df[(por_isp_df["variante"] == "cooperacao") & (por_isp_df["k"] > 0) & (por_isp_df["k"] < I)]
tabela = pc.groupby(["isp_id", "em_K"])["eta_i"].mean().unstack("em_K")
tabela.columns = ["eta_isolado" if not c else "eta_em_K" for c in tabela.columns]
tabela["diferenca"] = tabela["eta_em_K"] - tabela["eta_isolado"]
ff_isp = por_isp_df[por_isp_df["variante"] == "first_fit"].groupby("isp_id")["eta_i"].mean()
tabela["eta_first_fit"] = ff_isp
tabela.round(4)

## Conjuntos com maior e menor $\Gamma$

In [ ]:
por_conjunto = (
    coop.dropna(subset=["Gamma"])
    .groupby(["K", "k"])[["Gamma", "eta_K", "eta_isolados", "eta_todos", "B_n"]]
    .mean()
    .round(4)
    .sort_values("Gamma", ascending=False)
)
display(por_conjunto.head(10))
display(por_conjunto.tail(10))

## Rodadas idênticas

Conjuntos que produziram exatamente as mesmas métricas: mostra quando a cooperação não mudou nada na simulação.

In [ ]:
assinatura = coop.set_index(["base_index", "K"])[["eta_todos", "B_n", "B_m"]].round(6)
grupos = assinatura.reset_index().groupby(["base_index", "eta_todos", "B_n", "B_m"])["K"].apply(list)
print(f"{len(assinatura)} rodadas → {len(grupos)} resultados distintos")
for (base, *_), ks in grupos.items():
    print(f"base {base}: {', '.join(ks)}")

## Leitura

- **First-Fit vs ponderado com $K=\emptyset$:** efeito de ser disaster-aware, sem cooperação.
- **$K=\emptyset$ vs $K=\mathcal{I}$:** efeito de todos cooperarem.
- **$|K|=1$** tem de coincidir com $K=\emptyset$; se não, algo além da troca de informação está mudando.
- **$\Gamma(K) > 0$** só vale como resultado se aparecer de forma consistente entre conjuntos e cenários base. Com um cenário base só, trate como indicativo.